In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

In [27]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [3]:
# retail_sales_data = pd.read_csv("Fashion_Retail_Sales.csv")
# retail_sales_data.head()

In [4]:
inventory_data = pd.read_csv("retail_store_inventory_data.csv")
inventory_data.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [5]:
inventory_data.shape

(73100, 15)

In [6]:
inventory_data.columns = inventory_data.columns.str.lower().str.replace(" ", "_")
inventory_data.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday/promotion,competitor_pricing,seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [7]:
type(inventory_data)

pandas.core.frame.DataFrame

In [8]:
inventory_data['date'] = pd.to_datetime(inventory_data['date'])

In [9]:
inventory_data = inventory_data.sort_values(['product_id', 'date'])

In [10]:
inventory_data.columns

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday/promotion',
       'competitor_pricing', 'seasonality'],
      dtype='object')

In [11]:
inventory_data = inventory_data.rename(columns={'units_sold': 'sales'})

In [12]:
# On average, how many units of this product are sold per day over the last 7 days
inventory_data['sales_velocity_7d'] = (
    inventory_data.groupby('product_id')['sales']
    .rolling(7)
    .mean()
    .reset_index(level=0, drop=True)
)

In [17]:
inventory_data.set_index(['date'], inplace=True)

In [20]:
inventory_data['inventory_coverage_days'] = inventory_data['inventory_level'] / (inventory_data['sales_velocity_7d'] + 1e-5)

In [21]:
inventory_data['sales_velocity_14d'] = (
    inventory_data.groupby('product_id')['sales']
    .rolling(14)
    .mean()
    .reset_index(level=0, drop=True)
)

inventory_data['trend_score'] = inventory_data['sales_velocity_7d'] - inventory_data['sales_velocity_14d']

In [22]:
def recommendation(row):
    if row['trend_score'] > 0 and row['inventory_coverage_days'] < 5:
        return "Promote & Restock"
    elif row['inventory_coverage_days'] > 20:
        return "Deprioritize"
    else:
        return "Monitor"

inventory_data['recommendation'] = inventory_data.apply(recommendation, axis=1)

In [23]:
inventory_data['recommendation']

date
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
                    ...        
2024-01-01              Monitor
2024-01-01              Monitor
2024-01-01    Promote & Restock
2024-01-01    Promote & Restock
2024-01-01    Promote & Restock
Name: recommendation, Length: 73100, dtype: object

# gen ai layer

In [24]:
sample = inventory_data.iloc[-1]  # or filter for a specific product

context = f"""
Product: {sample['product_id']}
Sales velocity (7d): {sample['sales_velocity_7d']:.2f}
Trend score: {sample['trend_score']:.2f}
Inventory coverage: {sample['inventory_coverage_days']:.2f} days
Recommendation: {sample['recommendation']}
"""

In [25]:
prompt = f"""
You are a retail merchandising assistant.

Given the following data:
{context}

Explain:
1. Why this recommendation makes sense
2. What action should be taken
Keep it concise and business-focused.
"""

In [28]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

1. **Why this recommendation makes sense:**
   - The sales velocity of 140.29 indicates that the product is selling quickly, suggesting strong demand. The trend score of 20.29 shows positive momentum in sales, reinforcing that this item is gaining traction. However, the inventory coverage of only 0.83 days indicates that stock is running low, which could lead to missed sales opportunities if not addressed. Thus, promoting the product can boost visibility and drive sales further, while restocking is essential to meet ongoing demand.

2. **What action should be taken:**
   - Implement a targeted promotional campaign to enhance product visibility and increase sales. Simultaneously, initiate a restock order to replenish inventory to avoid stockouts and ensure sufficient supply to meet demand in the near term.
